# Fundamental Matrix Estimation: Why Normalization Matters

In this practice, you will implement the **8-point algorithm** for estimating the Fundamental matrix **F** — first **without** normalization, then **with** Hartley normalization, and finally compare them against OpenCV's built-in implementation.

By the end of this practice, you will understand:
- Why directly solving for **F** from pixel coordinates produces numerically unstable results
- How Hartley normalization fixes this by conditioning the data matrix
- How to verify the quality of **F** using the epipolar constraint

### Due date
2026/4/21  23:59

### Grading
Each practice has ***totally 20 points***. You have to complete objectives to get points.<br>
<span style="color:red">***Primary objective***</span> : basic objective you must complete.<br>
<span style="color:orange">***Bonus objective***</span> : difficulty objective for other bonus points.<br>
This practice has 6 <span style="color:red">***Primary objectives***</span>.

## Dataset

We will use **8 point correspondences** taken from the lecture slide example (two views of a house scene).
These are **pixel coordinates** from a 1376×1172 image pair.

| Point | Left image (u, v) | Right image (u', v') |
|:---:|:---:|:---:|
| #1 | (768, 76)   | (839, 47)   |
| #2 | (590, 249)  | (598, 168)  |
| #3 | (1249, 143) | (1325, 229) |
| #4 | (722, 679)  | (576, 594)  |
| #5 | (1141, 478) | (805, 512)  |
| #6 | (736, 881)  | (607, 798)  |
| #7 | (1123, 862) | (845, 924)  |
| #8 | (1275, 740) | (1209, 897) |

Run the cell below to load the data.

In [ ]:
import numpy as np

# Point correspondences from the lecture example
pts1 = np.array([
    [ 768,  76],
    [ 590, 249],
    [1249, 143],
    [ 722, 679],
    [1141, 478],
    [ 736, 881],
    [1123, 862],
    [1275, 740],
], dtype=float)

pts2 = np.array([
    [ 839,  47],
    [ 598, 168],
    [1325, 229],
    [ 576, 594],
    [ 805, 512],
    [ 607, 798],
    [ 845, 924],
    [1209, 897],
], dtype=float)

print('pts1 shape:', pts1.shape)
print('pts2 shape:', pts2.shape)

---
## Part 1: Without Normalization

We first build the data matrix **A** directly from pixel coordinates and solve for **F**.
Recall that the epipolar constraint $\mathbf{x}'^T \mathbf{F} \mathbf{x} = 0$ gives us one equation per point pair:

$$
\text{row}_i = [\,u'_i u_i,\; u'_i v_i,\; u'_i,\; v'_i u_i,\; v'_i v_i,\; v'_i,\; u_i,\; v_i,\; 1\,]
$$

Stacking all rows gives matrix **A** (8×9), and we solve $\mathbf{A}\mathbf{f} = \mathbf{0}$ via SVD.

The solution is the **last row of Vᵀ** (corresponding to the smallest singular value), reshaped to 3×3.

After obtaining **F**, we must enforce the **rank-2 constraint**: set the smallest singular value of **F** to zero.


<span style="color:red">***Primary objective (1/6, 3 points):***</span> Build matrix **A** from pixel coordinates without normalization (```A_unnorm```), then compute and print its **condition number** (```cond_unnorm```).

The condition number of **A** is defined as:
$$\kappa(A) = \frac{\sigma_{\max}}{\sigma_{\min}}
$$
where $\sigma_{\max}$ and $\sigma_{\min}$ are the largest and smallest singular values.
A very large condition number (e.g., > 10⁶) means the matrix is ill-conditioned.

Hint: `np.linalg.cond(A)` computes the condition number directly.

In [ ]:
## YOUR IMPLEMENTATION ##

A_unnorm = None   # Your code here
cond_unnorm = None# Your code here

## YOUR IMPLEMENTATION ##

print('A shape:', A_unnorm.shape)
print('Condition number of A (without normalization):', cond_unnorm)

<span style="color:red">***Primary objective (2/6, 3 points):***</span> Solve for **F** from `A_unnorm` using SVD, then enforce the rank-2 constraint.

Steps:
1. Compute SVD of `A_unnorm` → take the last row of **Vᵀ**, reshape to 3×3 → `F_unnorm_raw`
2. Compute SVD of `F_unnorm_raw` → set the smallest singular value to 0 → `F_unnorm`
3. Normalize scale so that `F_unnorm[2, 2] = 1`

In [ ]:
## YOUR IMPLEMENTATION ##

F_unnorm = None # Your code here

## YOUR IMPLEMENTATION ##

print('F (without normalization):')
print(F_unnorm)

---
## Part 2: With Hartley Normalization

Hartley normalization transforms the point coordinates **before** building **A**, so that all entries of **A** are at the same order of magnitude.

The transformation **T** for a set of points is:

$$
T = \begin{bmatrix} s & 0 & -s \cdot c_x \\ 0 & s & -s \cdot c_y \\ 0 & 0 & 1 \end{bmatrix}
$$

where:
- $(c_x, c_y)$ is the **centroid** of all points
- $s = \dfrac{\sqrt{2}}{d_{\text{mean}}}$, and $d_{\text{mean}}$ is the **mean distance** from the centroid

After normalization, the estimated $\hat{\mathbf{F}}$ lives in the normalized space. You must **de-normalize** at the end:

$$
\mathbf{F} = T'^\top \hat{\mathbf{F}}\, T
$$

> ⚠️ **The de-normalization step is the most commonly forgotten part. Do not skip it.**

<span style="color:red">***Primary objective (3/6, 3 points):***</span> Implement the `normalize_points` function.

It should:
1. Compute the centroid $(c_x, c_y)$ of the input points
2. Translate points so the centroid is at the origin
3. Compute the mean distance from the origin after translation
4. Compute scale $s = \sqrt{2}\, /\, d_{\text{mean}}$
5. Build the normalization matrix **T** (3×3)
6. Return the normalized points (N×2) and **T**

In [ ]:
def normalize_points(pts):
    """
    Apply Hartley normalization to a set of 2D points.

    Args:
        pts: (N, 2) array of pixel coordinates

    Returns:
        pts_norm: (N, 2) array of normalized coordinates
        T:        (3, 3) normalization matrix
    """
    ## YOUR IMPLEMENTATION ##

    T = None # Your code here             
    pts_norm = None # Your code here      
    
    ## YOUR IMPLEMENTATION ##
    return pts_norm, T


# Test your function
pts1_n, T1 = normalize_points(pts1)
pts2_n, T2 = normalize_points(pts2)

print('T1 (normalization matrix for image 1):')
print(T1)
print('Mean distance of pts1_n from origin (should be close to √2 ≈ 1.414):')
print(np.sqrt((pts1_n**2).sum(axis=1)).mean())
print('T2 (normalization matrix for image 1):')
print(T2)
print('Mean distance of pts2_n from origin (should be close to √2 ≈ 1.414):')
print(np.sqrt((pts2_n**2).sum(axis=1)).mean())

<span style="color:red">***Primary objective (4/6, 3 points):***</span> Use your `normalize_points` function to implement the full **normalized 8-point algorithm**.

Steps:
1. Normalize `pts1` and `pts2` using your function → get `pts1_n, T1` and `pts2_n, T2`
2. Build matrix **A** from the **normalized** coordinates
3. Solve via SVD → enforce rank-2 → `F_hat`
4. De-normalize: `F_norm = T2.T @ F_hat @ T1`
5. Normalize scale so that `F_norm[2, 2] = 1`
6. Print the condition number of the normalized **A** and compare it with the unnormalized one from Part 1

In [ ]:
## YOUR IMPLEMENTATION ##

# Step 1: Normalize
pts1_n, T1 = normalize_points(pts1)
pts2_n, T2 = normalize_points(pts2)

A_norm = None # Your code here
cond_norm = None # Your code here
F_norm = None # Your code here

## YOUR IMPLEMENTATION ##

print('Condition number of A (without normalization):', cond_unnorm)
print('Condition number of A (with normalization):   ', cond_norm)
print()
print('F (with normalization):')
print(F_norm)

---
## Part 3: Comparison

A correct **F** must satisfy the **epipolar constraint** for all corresponding point pairs:

$$
\mathbf{x}'^T \mathbf{F}\, \mathbf{x} = 0
$$

In practice, this value will not be exactly zero due to noise. But a good **F** should give values very close to zero (on the order of 1e-10 or smaller for perfect correspondences). A bad **F** will give large values.

We call $|\mathbf{x}'^T \mathbf{F}\, \mathbf{x}|$ the **algebraic error** for each point pair.

<span style="color:red">***Primary objective (5/6, 4 points):***</span> Compute the algebraic error $|\mathbf{x}'^T \mathbf{F}\, \mathbf{x}|$ for **all 8 point pairs**, for both `F_unnorm` and `F_norm`.

Steps:
1. Convert `pts1` and `pts2` to homogeneous coordinates (append a column of ones)
2. For each point pair, compute `abs(x2 @ F @ x1)` for both F matrices
3. Print the errors side by side and observe the difference

In [ ]:
## YOUR IMPLEMENTATION ##

# Step 1: Convert to homogeneous coordinates
pts1_h = None   # Your code here — shape (8, 3)
pts2_h = None   # Your code here — shape (8, 3)

# Step 2 & 3: Compute and print algebraic errors
print(f"{'Point':<8} {'Error (no norm)':>20} {'Error (with norm)':>20}")
print('-' * 52)
for i in range(8):
    err_unnorm = None   # Your code here
    err_norm   = None   # Your code here
    print(f"#{i+1:<7} {err_unnorm:>20.6e} {err_norm:>20.6e}")

## YOUR IMPLEMENTATION #

---
## Part 4: OpenCV implementation
<span style="color:red">***Primary objective (6/6, 4 points):***</span> 
Use OpenCV's built-in `cv2.findFundamentalMat()` to estimate **F**, then compute its algebraic errors and compare all three results.

`cv2.findFundamentalMat` uses the normalized 8-point algorithm internally and applies RANSAC for robustness.
Since all 8 of our points are correct correspondences (no outliers), you can use `method=cv2.FM_8POINT` to disable RANSAC.

```python
F_cv, mask = cv2.findFundamentalMat(pts1, pts2, method=cv2.FM_8POINT)
```

Compare `F_cv` with your `F_norm`. They should be very similar (up to a scale factor).

In [ ]:
import cv2

## YOUR IMPLEMENTATION ##

# Estimate F using OpenCV
F_cv = None   # Your code here

# Normalize scale so F_cv[2,2] = 1 for fair comparison
F_cv = None   # Your code here

# Print all three F matrices
print('F (without normalization):')
print(F_unnorm)
print()
print('F (your Hartley normalization):')
print(F_norm)
print()
print('F (OpenCV FM_8POINT):')
print(F_cv)
print()

# Compare algebraic errors for all three
print(f"{'Point':<8} {'No norm':>15} {'Hartley':>15} {'OpenCV':>15}")
print('-' * 58)
for i in range(8):
    err_unnorm = None   # Your code here
    err_norm   = None   # Your code here
    err_cv     = None   # Your code here
    print(f"#{i+1:<7} {err_unnorm:>15.6e} {err_norm:>15.6e} {err_cv:>15.6e}")

## YOUR IMPLEMENTATION ##